## Setup

Imports, project paths, and the API token. Same PROJECT_ROOT as
Football_Data_Extraction.ipynb, so both notebooks point at the same
data lake.

In [1]:
import os
import re
import time
import requests
import pandas as pd
import pyarrow as pa
from pathlib import Path
import sys

# Same project root as Football_Data_Extraction.ipynb
PROJECT_ROOT = Path.cwd()
sys.path.append(str(PROJECT_ROOT / "src"))

from utils.schema import TEAMS_MASTER_SCHEMA
from utils.parquet_io import read_partition, write_partition

API_TOKEN = os.environ.get("FOOTBALL_DATA_TOKEN")
if not API_TOKEN:
    raise ValueError("Token not found in environment variables.")

HEADERS = {"X-Auth-Token": API_TOKEN}
TEAMS_MASTER_PATH = PROJECT_ROOT / "data" / "reference" / "teams_master.parquet"
SCHEMA_COLS = [f.name for f in TEAMS_MASTER_SCHEMA]

## Helper functions

slugify() turns an API team name into the same team_key format
already used for the 5 promoted teams. fetch_teams() calls the API
for one league and returns its raw team list.

In [2]:
LEAGUE_META = {
    "laliga":        {"code": "PD",  "country": "Spain",    "flag": "ES"},
    "primeira_liga": {"code": "PPL", "country": "Portugal", "flag": "PT"},
}

def slugify(name: str) -> str:
    # Same convention already used for the 5 promoted teams:
    # lowercase, keep accents, non-word chars -> underscore
    return re.sub(r"[^\w]+", "_", name.lower(), flags=re.UNICODE).strip("_")

def fetch_teams(code: str) -> list[dict]:
    url = f"https://api.football-data.org/v4/competitions/{code}/teams"
    response = requests.get(url, headers=HEADERS, timeout=15)
    if response.status_code != 200:
        raise RuntimeError(f"Request failed for {code}: status {response.status_code} - {response.text}")
    return response.json()["teams"]

## Fetch all teams and split into matches vs new teams

For each team the API returns, check if its slug matches an existing team_key. If it matches, queue an update. If not, it is a brand new team to add.


In [3]:
existing = read_partition(TEAMS_MASTER_PATH).to_pandas()
existing = existing[SCHEMA_COLS].copy()
existing_keys = set(existing["team_key"])

updates = {}   # team_key -> (football_data_org_id, short_code)
new_rows = []

for league, meta in LEAGUE_META.items():
    teams = fetch_teams(meta["code"])
    print(f"{league} ({meta['code']}): {len(teams)} teams retrieved")
    for t in teams:
        key = slugify(t["name"])
        if key in existing_keys:
            updates[key] = (str(t["id"]), t.get("tla") or "")
            continue
        new_rows.append({
            "team_key": key,
            "official_name": t["name"],
            "short_code": t.get("tla") or "",
            "league": league,
            "country": meta["country"],
            "country_flag_iso2": meta["flag"],
            "football_data_org_id": str(t["id"]),
            "api_football_id": None,
            "is_promoted_2026_27": False,
        })
    time.sleep(6)

laliga (PD): 20 teams retrieved
primeira_liga (PPL): 18 teams retrieved


## Manual overrides for teams that do not slug-match

Some promoted teams use a shorter name in our master table than the API official name, so the automatic slug match misses them. These 4 are fixed here by hand, with their real football_data_org_id looked up manually. The check below stops the notebook if one of these ever starts matching automatically, so a manual override never silently overwrites a correct automatic match.


In [4]:
# Manual fixes: these promoted teams don't slug-match the API name exactly
MANUAL_ID_OVERRIDES = {
    "rc_deportivo":        ("560",  "DEP"),
    "real_racing_club":    ("5335", "SAN"),
    "marítimo":            ("5575", "CSM"),
    "académico_de_viseu":  ("5527", "ACV"),
}

for key, (fdid, short_code) in MANUAL_ID_OVERRIDES.items():
    if key in updates:
        raise ValueError(f"{key} ya matcheó automático, revisa antes de sobreescribir")
    updates[key] = (fdid, short_code)

## Review before writing anything

Print every promoted team with the ID it will get, plus how many brand new teams will be added. Check this output before running the next cell.


In [5]:
print("Coincidencias encontradas para los ascendidos (revisa antes de seguir):")
for key in existing_keys:
    if key in updates:
        print(f"  {key} -> football_data_org_id={updates[key][0]}, short_code={updates[key][1]}")
    else:
        print(f"  {key} -> SIN MATCH por nombre, revisar a mano")

print(f"\nEquipos nuevos a agregar: {len(new_rows)}")

Coincidencias encontradas para los ascendidos (revisa antes de seguir):
  casa_pia_ac -> football_data_org_id=6618, short_code=CAS
  sporting_clube_de_portugal -> football_data_org_id=498, short_code=SPO
  fc_famalicão -> football_data_org_id=5531, short_code=FAM
  cd_santa_clara -> football_data_org_id=5530, short_code=CD 
  sport_lisboa_e_benfica -> football_data_org_id=1903, short_code=BEN
  rc_celta_de_vigo -> football_data_org_id=558, short_code=CEL
  moreirense_fc -> football_data_org_id=583, short_code=MOR
  sporting_clube_de_braga -> football_data_org_id=5613, short_code=SCB
  fc_alverca -> football_data_org_id=7822, short_code=ALV
  athletic_club -> football_data_org_id=77, short_code=ATH
  ca_osasuna -> football_data_org_id=79, short_code=OSA
  málaga_cf -> football_data_org_id=84, short_code=MAL
  vitória_sc -> football_data_org_id=5543, short_code=GUI
  rio_ave_fc -> football_data_org_id=496, short_code=RIO
  gd_estoril_praia -> football_data_org_id=582, short_code=EST
  fc

## Write teams_master.parquet

Apply the updates to the existing rows, append the new teams, and overwrite the file with write_partition.

In [6]:
for key, (fdid, short_code) in updates.items():
    mask = existing["team_key"] == key
    existing.loc[mask, "football_data_org_id"] = fdid
    existing.loc[mask, "short_code"] = short_code

combined = pd.concat([existing, pd.DataFrame(new_rows)], ignore_index=True)[SCHEMA_COLS]
table = pa.Table.from_pandas(combined, schema=TEAMS_MASTER_SCHEMA, preserve_index=False)
write_partition(table, TEAMS_MASTER_PATH)
print(f"teams_master.parquet: {len(combined)} equipos totales ({len(new_rows)} nuevos)")

teams_master.parquet: 42 equipos totales (4 nuevos)


## Diagnostic and fix 4 duplicated promoted teams

Cell 7 wrote the file already, but the 4 teams that needed
`MANUAL_ID_OVERRIDES` (rc_deportivo, real_racing_club, marítimo,
académico_de_viseu) also slipped into `new_rows` under their full
API name, because the fetch loop only skips a team when its own
slug matches an existing `team_key` — it has no idea the manual
override even exists. Result: those 4 `football_data_org_id`
values are on disk twice, under two different `team_key` rows.

Run the diagnostic cell first and check the printed rows before
running the fix cell.


In [7]:
current = read_partition(TEAMS_MASTER_PATH).to_pandas()

dupe_ids = current["football_data_org_id"][current["football_data_org_id"].notna()]
dupe_ids = dupe_ids[dupe_ids.duplicated(keep=False)]
print("IDs duplicados encontrados:", sorted(dupe_ids.unique(), key=int))

to_drop = current[
    current["football_data_org_id"].isin(dupe_ids.unique()) & (~current["is_promoted_2026_27"])
]
print(to_drop[["team_key", "official_name", "football_data_org_id", "is_promoted_2026_27"]].to_string())
print(f"\nFilas a borrar: {len(to_drop)} | Filas actuales: {len(current)} | Filas resultantes: {len(current) - len(to_drop)}")


IDs duplicados encontrados: ['560', '5335', '5527', '5575']
                         team_key                  official_name football_data_org_id  is_promoted_2026_27
38         rc_deportivo_la_coruña         RC Deportivo La Coruña                  560                False
39  real_racing_club_de_santander  Real Racing Club de Santander                 5335                False
40          académico_de_viseu_fc          Académico de Viseu FC                 5527                False
41                    cs_marítimo                    CS Marítimo                 5575                False

Filas a borrar: 4 | Filas actuales: 42 | Filas resultantes: 38


### Fix only run after confirming the diagnostic output above

Should show 4 duplicated IDs (560, 5335, 5575, 5527), the 4 rows
to drop all with `is_promoted_2026_27=False`, and 38 resulting rows.


In [8]:
cleaned = current.drop(to_drop.index)[SCHEMA_COLS]
table = pa.Table.from_pandas(cleaned, schema=TEAMS_MASTER_SCHEMA, preserve_index=False)
write_partition(table, TEAMS_MASTER_PATH)
print(f"teams_master.parquet corregido: {len(cleaned)} equipos totales")


teams_master.parquet corregido: 38 equipos totales


## Add 16 historical (relegated) teams

teams_master only had the current-season roster (fetched via the
competitions/teams endpoint). These 16 ids appear in Bronze historical
seasons (2023-24/24-25/25-26) but never in 2026-27, so they were never
fetched. Reload Bronze + current teams_master to build the candidate list.

In [9]:
from utils.parquet_io import read_league_all_seasons

LAKE_ROOT = PROJECT_ROOT / "data"
laliga_bronze = read_league_all_seasons(LAKE_ROOT, layer="bronze", league="laliga").to_pandas()
primeira_bronze = read_league_all_seasons(LAKE_ROOT, layer="bronze", league="primeira_liga").to_pandas()

current_master = read_partition(TEAMS_MASTER_PATH).to_pandas()
known_ids = set(current_master["football_data_org_id"])

frames = []
for league, df in [("laliga", laliga_bronze), ("primeira_liga", primeira_bronze)]:
    home = df.loc[~df["home_team_id"].isin(known_ids), ["home_team", "home_team_id"]].rename(
        columns={"home_team": "official_name", "home_team_id": "football_data_org_id"})
    away = df.loc[~df["away_team_id"].isin(known_ids), ["away_team", "away_team_id"]].rename(
        columns={"away_team": "official_name", "away_team_id": "football_data_org_id"})
    combined = pd.concat([home, away]).drop_duplicates().assign(league=league)
    frames.append(combined)

candidates = pd.concat(frames).reset_index(drop=True)
print(f"{len(candidates)} equipos históricos a agregar")
candidates

16 equipos históricos a agregar


,official_name,football_data_org_id,league
0,UD Almería,267,laliga
1,UD Las Palmas,275,laliga
2,Cádiz CF,264,laliga
3,RCD Mallorca,89,laliga
4,Girona FC,298,laliga
5,Granada CF,83,laliga
6,Real Valladolid CF,250,laliga
7,CD Leganés,745,laliga
8,Real Oviedo,1048,laliga
9,SC Farense,5602,primeira_liga


## Fetch short_code for each historical team

Individual lookup (/v4/teams/{id}) since these teams aren't in the
current competition roster. 6s between calls, same rate limit as before.

In [15]:
def fetch_team_detail(team_id: str) -> dict:
    url = f"https://api.football-data.org/v4/teams/{team_id}"
    response = requests.get(url, headers=HEADERS, timeout=15)
    if response.status_code != 200:
        raise RuntimeError(f"Request failed for team {team_id}: status {response.status_code} - {response.text}")
    return response.json()

historical_new_rows = []
for _, row in candidates.iterrows():
    meta = LEAGUE_META[row["league"]]
    historical_new_rows.append({
        "team_key": slugify(row["official_name"]),
        "official_name": row["official_name"],
        "short_code": "",
        "league": row["league"],
        "country": meta["country"],
        "country_flag_iso2": meta["flag"],
        "football_data_org_id": row["football_data_org_id"],
        "api_football_id": None,
        "is_promoted_2026_27": False,
    })
    
print(len(historical_new_rows))
historical_new_rows[:3]

16


[{'team_key': 'ud_almería',
  'official_name': 'UD Almería',
  'short_code': '',
  'league': 'laliga',
  'country': 'Spain',
  'country_flag_iso2': 'ES',
  'football_data_org_id': '267',
  'api_football_id': None,
  'is_promoted_2026_27': False},
 {'team_key': 'ud_las_palmas',
  'official_name': 'UD Las Palmas',
  'short_code': '',
  'league': 'laliga',
  'country': 'Spain',
  'country_flag_iso2': 'ES',
  'football_data_org_id': '275',
  'api_football_id': None,
  'is_promoted_2026_27': False},
 {'team_key': 'cádiz_cf',
  'official_name': 'Cádiz CF',
  'short_code': '',
  'league': 'laliga',
  'country': 'Spain',
  'country_flag_iso2': 'ES',
  'football_data_org_id': '264',
  'api_football_id': None,
  'is_promoted_2026_27': False}]

In [16]:
new_keys = {r["team_key"] for r in historical_new_rows}
collisions = new_keys & set(current_master["team_key"])
print("Colisiones de team_key:", collisions if collisions else "ninguna")

for r in historical_new_rows:
    print(f"  {r['team_key']} -> id={r['football_data_org_id']}, short_code={r['short_code']}, league={r['league']}")

Colisiones de team_key: ninguna
  ud_almería -> id=267, short_code=, league=laliga
  ud_las_palmas -> id=275, short_code=, league=laliga
  cádiz_cf -> id=264, short_code=, league=laliga
  rcd_mallorca -> id=89, short_code=, league=laliga
  girona_fc -> id=298, short_code=, league=laliga
  granada_cf -> id=83, short_code=, league=laliga
  real_valladolid_cf -> id=250, short_code=, league=laliga
  cd_leganés -> id=745, short_code=, league=laliga
  real_oviedo -> id=1048, short_code=, league=laliga
  sc_farense -> id=5602, short_code=, league=primeira_liga
  boavista_fc -> id=810, short_code=, league=primeira_liga
  gd_chaves -> id=1103, short_code=, league=primeira_liga
  portimonense_sc -> id=5601, short_code=, league=primeira_liga
  fc_vizela -> id=5589, short_code=, league=primeira_liga
  avs -> id=10340, short_code=, league=primeira_liga
  cd_tondela -> id=1049, short_code=, league=primeira_liga


In [17]:
combined = pd.concat([current_master, pd.DataFrame(historical_new_rows)], ignore_index=True)[SCHEMA_COLS]
table = pa.Table.from_pandas(combined, schema=TEAMS_MASTER_SCHEMA, preserve_index=False)
write_partition(table, TEAMS_MASTER_PATH)
print(f"teams_master.parquet: {len(combined)} equipos totales ({len(historical_new_rows)} históricos agregados)")

teams_master.parquet: 54 equipos totales (16 históricos agregados)
